# Manufacturing Process Intelligence
## Predictive Task Assignment & Anomaly Detection for Automotive Assembly Lines

**Dataset:** Automotive assembly line event log — 143,308 rows, 2,869 vehicles, 2,050 unique tasks, 512 binary configuration features  
**Objective:**  
1. Given a vehicle's build configuration, predict which tasks it will require (_multi-label classification_)  
2. Identify vehicles whose actual task execution deviates from predictions (_anomaly detection_)

---
## 1. Setup & Data Loading

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

from src.data_loader import DataLoader

dl = DataLoader(filepath='../data/dec2jan12_mldata.csv')
dl.load().clean().pivot(min_task_support=0.05)
dl.summarize()

### Dataset Structure

| Dimension | Value |
|---|---|
| Raw rows | 143,308 |
| Unique vehicles | 2,869 |
| Config features | 512 binary columns |
| Unique tasks | 2,050 |
| Tasks kept (≥5% support) | 325 |
| Avg tasks per vehicle | 49.9 |

**Key insight:** The 512 binary config columns are constant per vehicle — they encode which options/variants the vehicle carries. The `CHARACTERISTIC` column records which assembly/inspection operations were performed. **Config determines process plan.**

In [ ]:
# Quick peek at the raw data
dl.raw_df[['SPNR8', 'CHARACTERISTIC', 'CHARACTERSTIC_DESCRIPTION']].head(8)

---
## 2. Exploratory Data Analysis

In [ ]:
tasks_per_v = dl.raw_df.groupby('SPNR8')['CHARACTERISTIC'].nunique()
vehicles_per_task = dl.raw_df.groupby('CHARACTERISTIC')['SPNR8'].nunique()
n_vehicles = dl.vehicle_ids.shape[0]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(tasks_per_v, bins=40, color='#4C72B0', edgecolor='white')
axes[0].axvline(tasks_per_v.mean(), color='red', linestyle='--',
                label=f'Mean={tasks_per_v.mean():.1f}')
axes[0].set_title('Tasks per Vehicle')
axes[0].set_xlabel('Number of Tasks'); axes[0].set_ylabel('Vehicles')
axes[0].legend()

pct = vehicles_per_task / n_vehicles * 100
axes[1].hist(pct, bins=50, color='#DD8800', edgecolor='white')
axes[1].axvline(5, color='red', linestyle='--', label='5% threshold (model cutoff)')
axes[1].set_title('Task Frequency Distribution')
axes[1].set_xlabel('% Vehicles with Task'); axes[1].set_ylabel('Tasks')
axes[1].legend()

plt.suptitle('EDA — Task Distribution', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Tasks in >50% of vehicles: {(pct > 50).sum()}')
print(f'Tasks in  >5% of vehicles: {(pct >= 5).sum()}  (kept for modelling)')
print(f'Tasks in  <5% of vehicles: {(pct < 5).sum()}  (too rare to learn)')

In [ ]:
# Config feature density
config_density = dl.vehicle_config.mean(axis=1) * 100

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(config_density, bins=30, color='#55AA88', edgecolor='white')
ax.axvline(config_density.mean(), color='red', linestyle='--',
           label=f'Mean = {config_density.mean():.1f}%')
ax.set_title('Vehicle Config Density — % Features Set to 1')
ax.set_xlabel('% Config Features ON'); ax.set_ylabel('Vehicles')
ax.legend()
plt.tight_layout()
plt.show()

---
## 3. Vehicle Clustering

Discover natural **variant families** from the 512-dimensional binary config space using PCA + KMeans, then visualise with UMAP.

In [ ]:
from src.clustering import VehicleClusterer

vc = VehicleClusterer(dl)
vc.run_pca(n_components=50)
vc.fit()   # auto-selects K via silhouette
cluster_labels = vc.get_cluster_labels()

print(f'\nCluster sizes:')
sizes = pd.Series(cluster_labels).value_counts().sort_index()
for cid, cnt in sizes.items():
    print(f'  Cluster {cid}: {cnt} vehicles ({cnt/len(cluster_labels)*100:.1f}%)')

In [ ]:
# Visualise PCA projection (first 2 PCs)
palette = sns.color_palette('tab10', 10)

fig, ax = plt.subplots(figsize=(9, 6))
for cid in range(vc.k):
    mask = cluster_labels == cid
    ax.scatter(vc.pca_coords[mask, 0], vc.pca_coords[mask, 1],
               s=15, alpha=0.6, label=f'Cluster {cid}', color=palette[cid])
ax.set_title('PCA — PC1 vs PC2, coloured by cluster')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.legend(markerscale=2, fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

In [ ]:
# UMAP (takes ~30s)
vc.run_umap()
print('UMAP complete — plot saved to outputs/figures/umap_vehicle_clusters.png')
from IPython.display import Image
Image('../outputs/figures/umap_vehicle_clusters.png')

---
## 4. Multi-label Task Prediction

**Problem:** Given a vehicle's 512 binary config → predict which of 325 common tasks it will receive.  
**Algorithm:** Binary Relevance — 325 independent XGBoost classifiers, one per task label.

In [ ]:
from src.multilabel_model import MultiLabelModel

ml = MultiLabelModel(dl, cluster_labels=cluster_labels)
ml.split()
ml.train_logistic()
ml.train_xgboost()

In [ ]:
ml.evaluate()
best_threshold = ml.tune_threshold()
print(f'\nBest threshold: {best_threshold}')

In [ ]:
Image('../outputs/figures/multilabel_model_comparison.png')

### Interpreting the Results

| Metric | Value | Interpretation |
|---|---|---|
| Micro Precision | **76.9%** | When the model predicts a task, it's correct 3 in 4 times |
| Micro F1 (threshold=0.20) | **28.4%** | 40% improvement over default 0.5 threshold |
| Hamming Loss | **0.0924** | 90.8% of all label predictions are correct |
| LRAP | **0.32** | Each true task is ranked in the top 32% of all 325 labels |

Low absolute F1 is expected: 325 labels, only 2,295 training vehicles, 9.84% label density.

In [ ]:
# Example: predict tasks for one vehicle
example_idx = 0
example_vid = dl.vehicle_ids[example_idx]
tasks, probs = ml.predict_vehicle(dl.vehicle_config[example_idx], threshold=best_threshold)

print(f'Vehicle: {example_vid}')
print(f'Predicted tasks: {len(tasks)}')
print('\nTop 10 highest-probability tasks:')
top10 = sorted(zip(dl.task_cols, probs), key=lambda x: -x[1])[:10]
for task, prob in top10:
    print(f'  {task:30s}  p={prob:.4f}')

---
## 5. Anomaly Detection

Two complementary signals:  
- **Isolation Forest** — unusual vehicle configs in PCA space  
- **Model Deviation** — vehicles whose actual tasks diverge from model predictions

In [ ]:
from src.anomaly_detection import AnomalyDetector

ad = AnomalyDetector(dl, ml, cluster_labels=cluster_labels)
ad.fit_isolation_forest(contamination=0.05)
ad.compute_deviation_scores(threshold=best_threshold)
ad.combine_scores()

In [ ]:
ad.plot_all()
report = ad.save_report()

print(f"\nFlagged at P95: {report['flagged_p95']} vehicles ({report['flagged_pct']}%)")
print(f"Max deviation : {max(e['deviation_tasks'] for e in report['top_20_anomalous_vehicles'])} tasks")

In [ ]:
Image('../outputs/figures/anomaly_deviation_vs_combined.png')

In [ ]:
# Top 10 anomalous vehicles
top10_df = pd.DataFrame(report['top_20_anomalous_vehicles'][:10])
top10_df.columns = ['Vehicle ID', 'Combined Score', 'Deviation', 'Unexpected Tasks', 'Missed Tasks']
top10_df

---
## 6. Summary

| Stage | Method | Key Result |
|---|---|---|
| Clustering | PCA (50 PCs, 94.2% var) + KMeans | 10 vehicle variant families, silhouette=0.38 |
| Multi-label | XGBoost Binary Relevance, 325 labels | 76.9% micro-precision, 28.4% F1 (tuned threshold) |
| Anomaly | Isolation Forest + Model Deviation | 144 of 2,869 vehicles flagged (5%), max deviation 118 tasks |

**Business value:** Pre-assigning tasks based on vehicle config enables proactive staffing and tool allocation before a vehicle reaches each station. Anomaly flagging surfaces process non-conformances and quality escapes in real time.